# DCN-V2 with MMoE

Issue: #27

## Question

Can a deep multi-task ranker, that predicts click, cart and order probabilities together, do better than LightGBM on the same candidates and features?

## Data used

The feature tables from #22 (`train_asof11.parquet`: 40,000 training queries, `eval_w0.parquet`: the 56,544 evaluation queries, 200 candidates each), the shared rules and metrics (#17), results of #18 and #23 for comparison.

Outputs: `data/models/dcn_mmoe/` (weights, config, feature scaling) and `data/results/dcn_mmoe.parquet`. Code: `notebooks/dcn.py`.

**Deviation from the issue text: no Two-Tower similarity input.** The issue lists the Two-Tower similarity as an input. The Two-Tower of #24 was trained on all sessions of the training window, which include the sessions of the ranker's training queries, so its score for a training row has already seen the label and would leak it (an evaluation row has not been seen). It also did not improve the pool (#26). So the model uses the 18 numeric features and the category of #22 only.

**Model.** 18 standardised numeric features plus an 8-dimensional category embedding form the input. Three DCN-V2 cross layers (`x_next = x0 * (W x + b) + x`) build feature interactions. Four expert networks read the cross output, and each of the three tasks (click, cart, order) has its own softmax gate that mixes the experts and its own tower that outputs a logit. **Loss** = weighted sum of per-task binary cross-entropy, weights click 1, cart 2, order 4 (orders are the rarest, so their loss would otherwise be small next to clicks). **Score** = `0.10 p_click + 0.30 p_cart + 0.60 p_order` (OTTO's weights), also tuned on validation.

In [ ]:
import json
import resource
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import paired_bootstrap_diff
from dcn import DCNMTL, OTTO_WEIGHTS, blend, ndcg10_pool, predict, task_auc, train_model
from evalset import TOP, evaluate, prepare_eval, summarize
from features import FEATURE_COLUMNS

torch.set_num_threads(4)
DATA = Path("data") if Path("data").exists() else Path("../data")
FEAT, RESULTS = DATA / "features", DATA / "results"
OUT = DATA / "models" / "dcn_mmoe"
OUT.mkdir(parents=True, exist_ok=True)
POOL, SEED = 200, 0
NUM = [c for c in FEATURE_COLUMNS if c not in ("category", "category_conf")]   # category_conf is constant 1.0
TASKS = ["y_click", "y_cart", "y_order"]
print(len(NUM), "numeric features:", NUM)

## Checks on the model first

1. Output shape is (batch, 3) and each task's gate weights sum to 1.
2. The model can overfit 100 examples (loss close to zero).

In [ ]:
torch.manual_seed(0)
m = DCNMTL(n_num=6, n_cats=5)
x = torch.randn(100, 6)
cat = torch.randint(0, 5, (100,))
y = (torch.rand(100, 3) < 0.3).float()
assert m(x, cat).shape == (100, 3)
x0 = torch.cat([x, m.emb(cat)], dim=-1)
for gate in m.gates:
    assert torch.allclose(torch.softmax(gate(x0), -1).sum(-1), torch.ones(100), atol=1e-5)
opt = torch.optim.Adam(m.parameters(), lr=0.01)
bce = torch.nn.BCEWithLogitsLoss()
first = bce(m(x, cat), y).item()
for _ in range(600):
    loss = bce(m(x, cat), y)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"overfit test: loss {first:.3f} -> {loss.item():.4f}")
assert loss.item() < 0.05 * first
ms = DCNMTL(n_num=6, n_cats=5, mode="shared")
assert ms(x, cat).shape == (100, 3)
print("model checks pass")

## Load the tables and prepare the tensors

Numeric features are standardised with the mean and standard deviation of the training rows. Sessions are split 90/10 into train and validation; the evaluation queries are never used for training or tuning.

In [ ]:
tr_all = pl.read_parquet(FEAT / "train_asof11.parquet")
sessions = tr_all["session"].unique().sort().to_numpy()
rng = np.random.default_rng(SEED)
valid_sessions = set(rng.choice(sessions, size=len(sessions) // 10, replace=False).tolist())
is_val = tr_all["session"].is_in(list(valid_sessions)).to_numpy()
assert not (set(tr_all.filter(~pl.Series(is_val))["session"].to_list()) & valid_sessions)

mean = tr_all.filter(~pl.Series(is_val)).select(NUM).mean().to_numpy().ravel().astype(np.float32)
std = tr_all.filter(~pl.Series(is_val)).select(NUM).std().to_numpy().ravel().astype(np.float32)
std[std < 1e-6] = 1.0


def tensors(df):
    x = torch.from_numpy((df.select(NUM).to_numpy().astype(np.float32) - mean) / std)
    c = torch.from_numpy(df["category"].to_numpy().astype(np.int64))
    y = torch.from_numpy(df.select(TASKS).to_numpy().astype(np.float32))
    return x, c, y, df["label"].to_numpy()


x_tr, c_tr, y_tr, lab_tr = tensors(tr_all.filter(~pl.Series(is_val)))
x_va, c_va, y_va, lab_va = tensors(tr_all.filter(pl.Series(is_val)))
del tr_all
print(f"train rows {len(x_tr):,} ({len(x_tr) // POOL:,} queries), validation rows {len(x_va):,} ({len(x_va) // POOL:,} queries)")
print("positive rate per task (train):", dict(zip(["click", "cart", "order"], y_tr.mean(0).numpy().round(5).tolist(), strict=True)))
n_cats = int(max(c_tr.max(), c_va.max())) + 1

## Train

In [ ]:
model = DCNMTL(n_num=len(NUM), n_cats=n_cats, n_tasks=3, mode="mmoe")
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")
history, best_epoch, seconds = train_model(model, x_tr, c_tr, y_tr, x_va, c_va, y_va, task_weights=(1.0, 2.0, 4.0), epochs=10, label="MMoE")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharex=True)
for t, (ax, name) in enumerate(zip(axes, ["click", "cart", "order"], strict=True)):
    ax.plot([h["epoch"] for h in history], [h["train_tasks"][t] for h in history], "o-", label="train")
    ax.plot([h["epoch"] for h in history], [h["val_tasks"][t] for h in history], "s--", label="validation")
    ax.set_title(f"{name} BCE"); ax.set_xlabel("epoch")
axes[0].legend()
plt.tight_layout(); plt.show()
assert history[-1]["train"] < history[0]["train"], "training loss did not decrease"

p_va = predict(model, x_va, c_va)
aucs = task_auc(p_va, y_va.numpy())
print("validation AUC per task (click, cart, order):", [round(a, 4) for a in aucs])
print("mean predicted probability vs positive rate (validation):")
for t, name in enumerate(["click", "cart", "order"]):
    print(f"  {name}: predicted {p_va[:, t].mean():.5f}, actual {y_va[:, t].mean().item():.5f}")
# the order task's loss is not swallowed by the others: its per-task loss decreased too
assert history[-1]["train_tasks"][2] < history[0]["train_tasks"][2] or best_epoch == 1

## Blend weights

The default score weights are OTTO's (0.10, 0.30, 0.60). A few alternatives are compared on the **validation** queries, using NDCG@10 inside the pool (not the evaluation queries).

In [ ]:
grid = [(0.10, 0.30, 0.60), (1.0, 0.0, 0.0), (0.34, 0.33, 0.33), (0.5, 0.3, 0.2), (0.2, 0.3, 0.5), (0.0, 0.5, 0.5), (0.05, 0.25, 0.70)]
rows = [{"weights": str(w), "val_ndcg10_in_pool": ndcg10_pool(blend(p_va, w), lab_va)} for w in grid]
tuning = pl.DataFrame(rows).sort("val_ndcg10_in_pool", descending=True)
print(tuning)
best_w = eval(tuning["weights"][0])
default_score = ndcg10_pool(blend(p_va, OTTO_WEIGHTS), lab_va)
print("best on validation:", best_w, "default OTTO weights:", round(default_score, 4))

## Evaluate on the 56,544 evaluation queries

In [ ]:
ev = pl.read_parquet(FEAT / "eval_w0.parquet")
prepared, prep_stats = prepare_eval(pl.read_parquet(DATA / "eval" / "ranking_queries.parquet"))
n_q = len(prepared)
assert ev.height == n_q * POOL
assert ev["session"].to_numpy()[::POOL].tolist() == [q["session"] for q in prepared]
x_ev, c_ev, y_ev, lab_ev = tensors(ev)
aids = ev["aid"].to_numpy().reshape(n_q, POOL)
row_of = {q["session"]: i for i, q in enumerate(prepared)}
p_ev = predict(model, x_ev, c_ev)
print("evaluation AUC per task:", [round(a, 4) for a in task_auc(p_ev, y_ev.numpy())])


def ranker(weights):
    scores = blend(p_ev, weights).reshape(n_q, POOL)
    order = np.argsort(-scores, axis=1, kind="stable")[:, :TOP]
    ranked = np.take_along_axis(aids, order, axis=1)
    return lambda q: [int(a) for a in ranked[row_of[q["session"]]]]


res_otto = evaluate(prepared, ranker(OTTO_WEIGHTS))
res_best = evaluate(prepared, ranker(best_w))


def show(name, r):
    s = summarize(r)
    print(name)
    for k, v in s.items():
        print(f"  {k:>22}: {v[0]:.4f}  [{v[1]:.4f}, {v[2]:.4f}]" if isinstance(v, tuple) else f"  {k:>22}: {v:.4f}  (OTTO official)")


show("DCN-V2 + MMoE, OTTO blend weights", res_otto)
if best_w != OTTO_WEIGHTS:
    show(f"DCN-V2 + MMoE, blend weights tuned on validation {best_w}", res_best)

## Compared with LightGBM (#23) and co-visitation (#18)

In [ ]:
lgbm = pl.read_parquet(RESULTS / "lgbm.parquet")
covis = pl.read_parquet(RESULTS / "covis.parquet")
for name, other in (("LightGBM lambdarank (#23)", lgbm), ("co-visitation (#18)", covis)):
    j = res_otto.join(other.select("session", pl.col("ndcg10").alias("o")), on="session")
    assert j.height == res_otto.height
    m_, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["o"].to_numpy())
    print(f"NDCG@10 DCN-V2+MMoE minus {name}: {m_:+.4f}  [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
torch.save(model.state_dict(), OUT / "model.pt")
(OUT / "config.json").write_text(json.dumps({
    "numeric_features": NUM, "n_cats": n_cats, "mode": "mmoe", "n_cross": 3, "n_experts": 4, "expert_dim": 64, "tower_dim": 32,
    "task_weights": [1.0, 2.0, 4.0], "blend_weights": list(OTTO_WEIGHTS), "best_val_blend": list(best_w), "epochs": 10, "best_epoch": best_epoch,
    "batch": 4096, "lr": 2e-3, "seed": SEED, "mean": mean.tolist(), "std": std.tolist(), "parameters": n_params, "train_seconds": round(seconds),
    "note": "no Two-Tower similarity input: the Two-Tower saw the training sessions (leak)",
}, indent=2))
res_otto.write_parquet(RESULTS / "dcn_mmoe.parquet")
assert (res_otto["n_returned"] == TOP).all()
print("saved model, config and results;", res_otto.height, "sessions; peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

Numbers are in the printouts above (56,544 evaluation queries, mean and 95% bootstrap interval).

| Model | NDCG@10 | recall@20 | recall@100 | OTTO weighted recall@20 |
|---|---|---|---|---|
| co-visitation + fill (#18) | 0.2144 | 0.3885 | 0.5553 | 0.3981 |
| LightGBM lambdarank (#23) | 0.2153 | 0.3913 | 0.5574 | 0.3928 |
| DCN-V2 + MMoE, OTTO blend (0.10, 0.30, 0.60) | 0.2069 | 0.3934 | 0.5574 | 0.3980 |
| DCN-V2 + MMoE, blend tuned on validation (0.34, 0.33, 0.33) | 0.2079 | 0.3943 | 0.5576 | 0.3926 |

- **DCN-V2 + MMoE does not beat LightGBM or co-visitation on NDCG@10.** Paired NDCG@10 difference against LightGBM: **-0.0084 [-0.0095, -0.0073]**; against co-visitation: -0.0075 [-0.0087, -0.0064]. Both intervals exclude zero, so it is slightly worse.
- **It finds slightly more in the top 20 but orders the top 10 worse.** Its recall@20 (0.393) and OTTO weighted recall@20 (0.398) are at or above LightGBM's (0.391, 0.393), while NDCG@10, which rewards the order at the very top, is lower.
- **Each task is learned reasonably.** Validation AUC per task (click, cart, order): 0.851, 0.905, 0.928; evaluation AUC: 0.865, 0.900, 0.892. Predicted mean probabilities are close to the real rates (click 0.0042 vs 0.0042, cart 0.00034 vs 0.00030, order 0.00006 vs 0.00008), so the probabilities are roughly calibrated. The order task's loss went down like the others (per-task validation loss 0.0022 to 0.0007), so its rare positives were not drowned out by the click loss.
- **Not under-trained.** A first run with 3 epochs (best epoch was the last one) gave NDCG@10 0.2075; letting it train for 10 epochs (best epoch 9, loss flat from epoch 8) gave 0.2069. More training did not help.
- **The blend weights barely matter.** Tuned on validation (not on the evaluation queries), the equal weights were best (0.34, 0.33, 0.33); on the evaluation queries this is +0.001 NDCG@10 over OTTO's weights. OTTO's weights are meant to match the competition metric, and our NDCG@10 gains (click 1, cart 2, order 3) weight the tasks differently, which is one reason equal weights work as well.
- **Why it does not win.** As for LightGBM (#23), the features are dominated by co-visitation signals (`rank_covis`, `covis_max`, ...). There is little for a deeper model to add on top of the order the pool already has, and the trees handle those sharp rank features at least as well as the cross layers. The model has only 33,000 parameters and was trained on 36,000 queries.
- **Not done:** a version with the Two-Tower similarity (leaks, see the top), hyperparameter tuning, a larger model, using the pool order as an explicit prior. #28 compares against single-task and shared-bottom versions of this network.

## What I learned

To be written by the owner of the project.